# Tổng hợp đánh giá từ `results/evaluations`

Notebook này load tất cả file `*_eval.json` trong `results/evaluations`, tính mean / variance / std cho các chỉ số chính, và xuất bảng tổng hợp cuối cùng.

In [9]:
import json
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd()
if not (ROOT / 'results' / 'evaluations').exists():
    alt = ROOT.parent
    if (alt / 'results' / 'evaluations').exists():
        ROOT = alt
    else:
        raise FileNotFoundError(
            f"Cannot find 'results/evaluations' in {ROOT} or {alt}.\n"
            "Vui lòng chạy notebook từ thư mục gốc của repo hoặc đảm bảo thư mục đánh giá tồn tại."
        )
EVALUATION_DIR = ROOT / 'results' / 'evaluations'

eval_files = sorted(EVALUATION_DIR.glob('*_eval.json'))
print('Found', len(eval_files), 'evaluation files in', EVALUATION_DIR)
for path in eval_files:
    print('-', path.name)

Found 12 evaluation files in d:\Github\mcs-train-content-model\results\evaluations
- md-gpt-5.4-mini_21-05-2026_13-53_eval.json
- md-gpt-5.4-mini_21-05-2026_14-31_eval.json
- qwen3.5-2b-facebook-content_nothinking_21-05-2026_21-36_eval.json
- qwen3.5-2b-facebook-content_nothinking_21-05-2026_21-44_eval.json
- qwen3.5-4b-facebook-content_nothinking_21-05-2026_23-16_eval.json
- qwen3.5-4b-facebook-content_nothinking_22-05-2026_06-05_eval.json
- qwen3.5_2b_nothinking_21-05-2026_13-35_eval.json
- qwen3.5_2b_nothinking_21-05-2026_14-01_eval.json
- qwen3.5_4b_nothinking_21-05-2026_17-18_eval.json
- qwen3.5_4b_nothinking_21-05-2026_17-33_eval.json
- qwen3.5_9b_nothinking_21-05-2026_19-36_eval.json
- qwen3.5_9b_nothinking_21-05-2026_20-11_eval.json


In [10]:
records = []
for path in eval_files:
    data = json.loads(path.read_text(encoding='utf-8'))
    model_name = path.name.replace('_eval.json', '')
    for item in data:
        records.append({
            'model': model_name,
            'faithfulness': item.get('faithfulness_combined'),
            'expansion': item.get('expansion_combined'),
            'vibe': item.get('vibe_combined'),
        })

df = pd.DataFrame(records)
if df.empty:
    raise ValueError(
        f"No evaluation records were loaded. Check eval_files count = {len(eval_files)} "
        "and ensure the notebook is running from the repository root."
    )
for col in ['faithfulness', 'expansion', 'vibe']:
    if col not in df.columns:
        raise KeyError(f"Missing expected column '{col}' in DataFrame columns={df.columns.tolist()}")

df[['faithfulness', 'expansion', 'vibe']] = df[['faithfulness', 'expansion', 'vibe']].astype(float)
df['overall'] = df[['faithfulness', 'expansion', 'vibe']].mean(axis=1)

grouped = df.groupby('model')
summary = grouped.agg([np.mean, np.var, np.std])
summary.columns = ['_'.join(col).strip() for col in summary.columns.values]
summary = summary.reset_index()

summary[['faithfulness_mean', 'faithfulness_var', 'faithfulness_std',
         'expansion_mean', 'expansion_var', 'expansion_std',
         'vibe_mean', 'vibe_var', 'vibe_std',
         'overall_mean', 'overall_var', 'overall_std']] = summary[[
             'faithfulness_mean', 'faithfulness_var', 'faithfulness_std',
             'expansion_mean', 'expansion_var', 'expansion_std',
             'vibe_mean', 'vibe_var', 'vibe_std',
             'overall_mean', 'overall_var', 'overall_std'
         ]]
summary = summary.sort_values('overall_mean', ascending=False)

summary_round = summary.round(4)
summary_round

,model,faithfulness_mean,faithfulness_var,faithfulness_std,expansion_mean,expansion_var,expansion_std,vibe_mean,vibe_var,vibe_std,overall_mean,overall_var,overall_std
1,md-gpt-5.4-mini_21-05-2026_14-31,0.6588,0.0096,0.0982,0.8170,0.0047,0.0689,0.6451,0.0065,0.0804,0.7069,0.0030,0.0547
0,md-gpt-5.4-mini_21-05-2026_13-53,0.6794,0.0101,0.1005,0.7785,0.0081,0.0901,0.6477,0.0076,0.0870,0.7019,0.0039,0.0626
5,qwen3.5-4b-facebook-content_nothinking_22-05-2...,0.6366,0.0090,0.0950,0.7766,0.0050,0.0706,0.6482,0.0073,0.0852,0.6871,0.0027,0.0518
4,qwen3.5-4b-facebook-content_nothinking_21-05-2...,0.6329,0.0160,0.1264,0.7663,0.0030,0.0543,0.6513,0.0063,0.0792,0.6835,0.0037,0.0611
3,qwen3.5-2b-facebook-content_nothinking_21-05-2...,0.5970,0.0142,0.1191,0.7542,0.0044,0.0665,0.5866,0.0066,0.0815,0.6459,0.0041,0.0636
10,qwen3.5_9b_nothinking_21-05-2026_19-36,0.6002,0.0139,0.1180,0.7914,0.0034,0.0583,0.5370,0.0082,0.0905,0.6429,0.0034,0.0585
2,qwen3.5-2b-facebook-content_nothinking_21-05-2...,0.6075,0.0117,0.1083,0.7140,0.0137,0.1171,0.5937,0.0105,0.1027,0.6384,0.0059,0.0765
11,qwen3.5_9b_nothinking_21-05-2026_20-11,0.5844,0.0132,0.1150,0.7803,0.0046,0.0677,0.5381,0.0078,0.0882,0.6343,0.0040,0.0633
8,qwen3.5_4b_nothinking_21-05-2026_17-18,0.5954,0.0111,0.1053,0.7866,0.0026,0.0506,0.5083,0.0064,0.0798,0.6301,0.0030,0.0550
9,qwen3.5_4b_nothinking_21-05-2026_17-33,0.5958,0.0148,0.1218,0.7680,0.0042,0.0649,0.5016,0.0059,0.0770,0.6218,0.0044,0.0662


In [12]:
from IPython.display import HTML, display

final_table = summary[['model', 'faithfulness_mean', 'expansion_mean', 'vibe_mean', 'overall_mean']].copy()
final_table.columns = ['model', 'Faith', 'Expan', 'Vibe', 'Overall']
final_table = final_table.round(4)
display(HTML(final_table.to_html(index=False)))

model,Faith,Expan,Vibe,Overall
md-gpt-5.4-mini_21-05-2026_14-31,0.6588,0.8170,0.6451,0.7069
md-gpt-5.4-mini_21-05-2026_13-53,0.6794,0.7785,0.6477,0.7019
qwen3.5-4b-facebook-content_nothinking_22-05-2026_06-05,0.6366,0.7766,0.6482,0.6871
qwen3.5-4b-facebook-content_nothinking_21-05-2026_23-16,0.6329,0.7663,0.6513,0.6835
qwen3.5-2b-facebook-content_nothinking_21-05-2026_21-44,0.5970,0.7542,0.5866,0.6459
qwen3.5_9b_nothinking_21-05-2026_19-36,0.6002,0.7914,0.5370,0.6429
qwen3.5-2b-facebook-content_nothinking_21-05-2026_21-36,0.6075,0.7140,0.5937,0.6384
qwen3.5_9b_nothinking_21-05-2026_20-11,0.5844,0.7803,0.5381,0.6343
qwen3.5_4b_nothinking_21-05-2026_17-18,0.5954,0.7866,0.5083,0.6301
qwen3.5_4b_nothinking_21-05-2026_17-33,0.5958,0.7680,0.5016,0.6218
